In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,classification_report
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier,StackingClassifier
from xgboost import XGBClassifier


In [2]:
df=pd.read_csv("C:\\Users\\Debomoy Patra\\Downloads\\archive\\kidney_disease_dataset.csv")
df.head()
df.columns
df.dtypes


Age of the patient                               int64
Blood pressure (mm/Hg)                           int64
Specific gravity of urine                      float64
Albumin in urine                                 int64
Sugar in urine                                   int64
Red blood cells in urine                        object
Pus cells in urine                              object
Pus cell clumps in urine                        object
Bacteria in urine                               object
Random blood glucose level (mg/dl)               int64
Blood urea (mg/dl)                             float64
Serum creatinine (mg/dl)                       float64
Sodium level (mEq/L)                           float64
Potassium level (mEq/L)                        float64
Hemoglobin level (gms)                         float64
Packed cell volume (%)                           int64
White blood cell count (cells/cumm)              int64
Red blood cell count (millions/cumm)           float64
Hypertensi

In [3]:
categorical_columns = ['Red blood cells in urine', 'Pus cells in urine', 'Pus cell clumps in urine', 
                      'Bacteria in urine', 'Hypertension (yes/no)', 'Diabetes mellitus (yes/no)', 
                      'Coronary artery disease (yes/no)', 'Appetite (good/poor)', 'Pedal edema (yes/no)', 
                      'Anemia (yes/no)', 'Family history of chronic kidney disease', 'Smoking status', 
                      'Physical activity level', 'Urinary sediment microscopy results']

binary_columns = ['Hypertension (yes/no)', 'Diabetes mellitus (yes/no)', 'Coronary artery disease (yes/no)', 
                  'Pedal edema (yes/no)', 'Anemia (yes/no)', 'Family history of chronic kidney disease', 
                  'Smoking status']

for i in binary_columns:
    df[i]=df[i].map({'yes':1,'no':0})
df['Appetite'] = df['Appetite (good/poor)'].map({'good': 1, 'poor': 0})
df['Physical activity level'] = df['Physical activity level'].map({'low': 0, 'moderate': 1, 'high': 2})

label_encoders={}
for col in ['Red blood cells in urine', 'Pus cells in urine', 'Pus cell clumps in urine', 
            'Bacteria in urine', 'Urinary sediment microscopy results']:
    le=LabelEncoder()
    df[col]=le.fit_transform(df[col].astype(str))
    label_encoders[col]=le

In [4]:
from sklearn.model_selection import train_test_split
X=df.drop('Target',axis=1)

y=le.fit_transform(df['Target'].astype(str))


columns_to_drop = ['Appetite (good/poor)'] +['Physical activity level'] + categorical_columns
X = X.drop([col for col in columns_to_drop if col in X.columns], axis=1)
#splitting the data

X_test,X_train,y_test,y_train=train_test_split(X,y,test_size=0.2, random_state=42, shuffle= True)


In [5]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [6]:

model={
    "Logistic Regression":LogisticRegression(max_iter=5000,random_state=42,class_weight='balanced'),
    "Decision Tree":DecisionTreeClassifier(random_state=42),
    "SVM":SVC(random_state=42,kernel="rbf",class_weight='balanced'),
    "neural_network":MLPClassifier(random_state=42, max_iter=5000),
    "Random Forest":RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=42),
    "AdaBoost":AdaBoostClassifier(estimator=None, n_estimators=50, learning_rate=1.0, random_state=None),
    "Xgboost":XGBClassifier(n_estimators=100,max_depth=5,learning_rate=0.1,eval_metric="logloss")
}



In [7]:
accuracy_res = []

for i, m in model.items():
    if i in ['Logistic Regression', 'SVM','neural_network']:
        m.fit(X_train_scaled, y_train)
        y_pred = m.predict(X_test_scaled)
    # elif i in ['Xgboost']:
    #     y_train=scaler.fit_transform(y_train)
    #     y_test=scaler.transform(y_test)
    #     m.fit(X_train_scaled, y_train)
    #     y_pred = m.predict(X_test_scaled)
    else:
        m.fit(X_train, y_train)
        y_pred = m.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracy_res.append((i, accuracy))

    print("Classififcation Report")
    print(i)
    print(f"Accuracy: {accuracy * 100:.2f}%")


Classififcation Report
Logistic Regression
Accuracy: 17.64%
Classififcation Report
Decision Tree
Accuracy: 62.06%
Classififcation Report
SVM
Accuracy: 51.09%
Classififcation Report
neural_network
Accuracy: 65.87%
Classififcation Report
Random Forest
Accuracy: 79.95%


c:\Users\Debomoy Patra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Classififcation Report
AdaBoost
Accuracy: 79.64%
Classififcation Report
Xgboost
Accuracy: 79.91%


In [8]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
    
for name, m in model.items():
    if name in ["Logistic Regression", "SVM", "Neural Network"]:
        X_data = X_train_scaled
    else:
        X_data = X_train
    scores = cross_val_score(
        m,
        X_data,
        y_train,
        cv=cv,
        scoring="accuracy"
    )
    
    print(name)
    print(f"CV Accuracy: {scores.mean()*100:.2f}%")
    print(f"Std: ±{scores.std()*100:.2f}%")

Logistic Regression
CV Accuracy: 17.62%
Std: ±1.35%
Decision Tree
CV Accuracy: 63.46%
Std: ±1.99%
SVM
CV Accuracy: 52.90%
Std: ±0.70%
neural_network
CV Accuracy: 69.05%
Std: ±20.41%
Random Forest
CV Accuracy: 80.23%
Std: ±0.05%


c:\Users\Debomoy Patra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Debomoy Patra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Debomoy Patra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\Debomoy Patra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAM

AdaBoost
CV Accuracy: 79.75%
Std: ±0.19%
Xgboost
CV Accuracy: 80.16%
Std: ±0.07%


In [9]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[    0     0     0   653     0]
 [    0     0     0  1638     0]
 [    0     0     0   666     0]
 [    0     5     1 13130     0]
 [    0     0     0   337     0]]


In [10]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       653
           1       0.00      0.00      0.00      1638
           2       0.00      0.00      0.00       666
           3       0.80      1.00      0.89     13136
           4       0.00      0.00      0.00       337

    accuracy                           0.80     16430
   macro avg       0.16      0.20      0.18     16430
weighted avg       0.64      0.80      0.71     16430



c:\Users\Debomoy Patra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Debomoy Patra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Debomoy Patra\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.